# DLinear and Transformer Split-Horizon Forecasting, Leakage-Safe Selector (mixed4)

This notebook uses one shared BasicTS forecasting data pipeline for both models. DLinear forecasts steps 1-6, the Transformer forecasts steps 7-12, and their 6-step outputs are concatenated into one 12-step hybrid forecast.

The selector section avoids test-set leakage: hard selector masks and soft weights are learned from validation predictions, then applied once to the test predictions for final reporting.

Another thing before next meeting is that you should think what challenges existing paper has, so we have to propose current method. just add what you need from the past MOE Tell me the background first, explain that you think should explain, and make the results clear

## 1. Project Setup

This cell keeps the notebook runnable from inside `notebooks/` by moving to the repo root and adding `src/` to Python's import path.


In [1]:
import os 
import sys
from pathlib import Path
#root contains the path to the basicts
ROOT = Path(r"C:\Users\luwil\OneDrive\Documents\Code\BasicTS")
#move python working folder
os.chdir(ROOT)
# the path to src is src_path
src_path = ROOT / "src"
#if src_path is not in the system path, add it to the system path
#system path is added to python search. 
# This allows us to import modules from the src 
# folder without having to specify the full path.
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

## 2. Imports and Shared Settings

Both models use the same dataset, scaler, preprocessing, `input_len`, train/val/test split, and batch format. The shared dataset keeps the full 12-step target window, while each split-horizon model trains on its own 6-step slice.


In [2]:
import json
from datetime import datetime
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader

from basicts.configs import BasicTSForecastingConfig, BasicTSModelConfig
from basicts.launcher import BasicTSLauncher
from basicts.models.DLinear import DLinear, DLinearConfig
from basicts.models.iTransformer import iTransformerConfig, iTransformerForForecasting
from basicts.runners.builder import Builder
from basicts.runners.taskflow import BasicTSForecastingTaskFlow
from basicts.scaler import ZScoreScaler
from basicts.utils import BasicTSMode
DATASET_NAME = "ETTh1"
#most papers use 96 input length
INPUT_LEN = 96
#most papers use 96, 192, 336, and 720 input length
FULL_OUTPUT_LEN = 12

SPLIT_OUTPUT_LEN = 6
#etthl has 7 variables
NUM_FEATURES = 7
#Batch sizes like 16, 32, and 64 are normal. 32 is a safe default.
BATCH_SIZE = 32
# common for testing
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3
# Fresh namespace for this notebook so BasicTS does not auto-resume from old/corrupt checkpoints.
RUN_TAG = "mixed_4"
# this is the shared settings dictionary that both models use
SHARED_CONFIG = {
    "dataset_name": DATASET_NAME,
    "input_len": INPUT_LEN,
    "dataset_params": {
        "input_len": INPUT_LEN,
        "output_len": FULL_OUTPUT_LEN,
        "use_timestamps": False,
        "memmap": False,
    },
    "use_timestamps": False,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "scaler": ZScoreScaler,
    "norm_each_channel": True,
    "rescale": False,
    "metrics": ["MAE", "MSE"],
    "optimizer_params": {"lr": LEARNING_RATE, "weight_decay": 5e-4},
    "gpus": None,
    "train_data_num_workers": 0,
    "val_data_num_workers": 0,
    "test_data_num_workers": 0,
    "save_results": True,
}

# The standalone 12-step models only need BasicTS test_metrics.json.
# Metrics-only evaluation avoids Windows memmap file-lock issues.
FULL_12_STEP_CONFIG = dict(SHARED_CONFIG)
FULL_12_STEP_CONFIG["save_results"] = False


def fresh_checkpoint_dir(model_folder, run_name):
    # Each training launch gets a unique parent folder, so BasicTS cannot resume a stale checkpoint.
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return str(Path("checkpoints") / RUN_TAG / model_folder / run_name / stamp)

## 3. Shared Shape Check Helper

This helper builds the BasicTS dataset and scaler, runs the same forecasting preprocessing that training uses, and then sends one batch through the selected model.


In [3]:
# for the split forecasitting model, we need to create a custom taskflow that slices the targets and target masks to the desired output length      
#start with the forecasting taskflwo and mofify it 
# BasicTSForecastingTaskFlow is the default data-prep worker.
# It prepares each forecasting batch before the model uses it.
# start with the deault taskflow and then add a change 
class SplitHorizonForecastingTaskFlow(BasicTSForecastingTaskFlow):
    #adding a setting called targest slide whchi is a variable that is used to slice the targets
    # tells the taskflow which targets to keep
    # need self because we need acresss to this specific object
    #creates a variables incide the class object 
    def __init__(self, target_slice):
        self.target_slice = target_slice
    # preprocess 
    def preprocess(self, runner, data):
        #normal work
        data = super().preprocess(runner, data)
        #cuts target values
        data["targets"] = data["targets"][:, self.target_slice, :]
        # tells basicts which target values are valid
        data["targets_mask"] = data["targets_mask"][:, self.target_slice, :]
        return data


def _float_batch(batch):
    return {
        key: value.float() if isinstance(value, torch.Tensor) and value.is_floating_point() else value
        for key, value in batch.items()
    }

# checks to see if the inputs are the right shape the targest are sliced and the prediction matches targer
def preview_shapes(cfg, model_name):
    #build the dataset using basicts
    train_dataset = Builder._build_dataset(cfg, BasicTSMode.TRAIN)
    # puts the dataset into batches gets ready for batches
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=False)
    #This grabs the first batch.
    raw_batch = _float_batch(next(iter(train_loader)))
    #creates the scaler and fits it to the training data  
    scaler = Builder._build_scaler(cfg)
    scaler.fit(train_dataset.data)
    # fake runner so the pasicts 
    class PreviewRunner:
        pass
    # Create a fake runner that has cfg and scaler, because taskflow.preprocess expects a runner object.
    runner = PreviewRunner() 
    runner.cfg = cfg
    runner.scaler = scaler
    #prepares the batch normally
    processed_batch = cfg.taskflow.preprocess(runner, dict(raw_batch))
    #build the model from the config and switch it to evaluation mode
    model = cfg.model(cfg.model_config)
    model.eval()
    # o not track gradient as they are only needed for trainig 
    with torch.no_grad():
        #sends processed inputs into the model
        prediction = model(processed_batch["inputs"])
        # then the model outputs future values
        # did the model return a dictionary
        if isinstance(prediction, dict):
            # if it did this extracts onlt the prediction tensor
            prediction = prediction["prediction"]
    #print the shapres to check that the data and model match before the training
    print(f"{model_name} raw inputs shape:       ", tuple(raw_batch["inputs"].shape))
    print(f"{model_name} raw target shape:       ", tuple(raw_batch["targets"].shape))
    print(f"{model_name} processed inputs shape: ", tuple(processed_batch["inputs"].shape))
    print(f"{model_name} target shape:           ", tuple(processed_batch["targets"].shape))
    print(f"{model_name} prediction shape:       ", tuple(prediction.shape))
    #checks
    assert tuple(raw_batch["targets"].shape) == (cfg.batch_size, FULL_OUTPUT_LEN, NUM_FEATURES)
    assert tuple(processed_batch["inputs"].shape) == (cfg.batch_size, INPUT_LEN, NUM_FEATURES)
    assert tuple(processed_batch["targets"].shape) == (cfg.batch_size, SPLIT_OUTPUT_LEN, NUM_FEATURES)
    assert tuple(prediction.shape) == (cfg.batch_size, SPLIT_OUTPUT_LEN, NUM_FEATURES)
    return processed_batch, prediction


## 4. DLinear Model

This notebook uses BasicTS's built-in `DLinear` model for forecast steps 1-6. `DLinear` receives `[batch_size, input_len, num_features]` and returns `[batch_size, 6, num_features]` for the split-horizon branch.

In [4]:
# DLinear is imported from BasicTS in the imports cell:
# from basicts.models.DLinear import DLinear, DLinearConfig
# No custom model class is needed for this branch.

## 5. DLinear Config

This config reuses `BasicTSForecastingConfig` and only changes the model-specific pieces. The shared dataset/scaler/preprocessing settings come from `SHARED_CONFIG`.


In [5]:
# tells BasicTS how to build and train DLinear
dlinear_model_config = DLinearConfig(
    input_len=INPUT_LEN,
    output_len=SPLIT_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    moving_avg=25,
    stride=1,
    individual=False,
)

# full BasicTS training config for split-horizon DLinear
dlinear_cfg = BasicTSForecastingConfig(
    model=DLinear,
    model_config=dlinear_model_config,
    taskflow=SplitHorizonForecastingTaskFlow(slice(0, SPLIT_OUTPUT_LEN)),
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/DLinear/{DATASET_NAME}_{INPUT_LEN}_steps_1_6",
    **SHARED_CONFIG,
)

# Standalone DLinear trained to forecast all 12 steps.
dlinear_full_model_config = DLinearConfig(
    input_len=INPUT_LEN,
    output_len=FULL_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    moving_avg=25,
    stride=1,
    individual=False,
)

dlinear_full_cfg = BasicTSForecastingConfig(
    model=DLinear,
    model_config=dlinear_full_model_config,
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/DLinear/{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    **FULL_12_STEP_CONFIG,
)

dlinear_cfg, dlinear_full_cfg

(BasicTSForecastingConfig(model=<class 'basicts.models.DLinear.arch.dlinear_arch.DLinear'>, model_config=DLinearConfig(input_len=96, output_len=6, num_features=7, moving_avg=25, stride=1, individual=False), dataset_name='ETTh1', taskflow=<__main__.SplitHorizonForecastingTaskFlow object at 0x0000023F03AAD250>, callbacks=[], gpus=None, gpu_num=0, seed=42, dataset_type=<class 'basicts.data.tsf_dataset.BasicTSForecastingDataset'>, dataset_params={'input_len': 96, 'output_len': 12, 'use_timestamps': False, 'memmap': False, 'dataset_name': 'ETTh1'}, batch_size=32, null_val=nan, null_to_num=0.0, scaler=<class 'basicts.scaler.z_score_scaler.ZScoreScaler'>, norm_each_channel=True, rescale=False, ddp_find_unused_parameters=False, compile_model=False, metrics=['MAE', 'MSE'], target_metric='MAE', best_metric='min', num_epochs=5, num_steps=None, loss='MAE', optimizer=<class 'torch.optim.adam.Adam'>, optimizer_params={'lr': 0.001, 'weight_decay': 0.0005}, lr=None, lr_scheduler=None, lr_scheduler_par

## 6. DLinear Shape Test

Run this before training. The model prediction and processed target lines must be `(batch_size, 6, num_features)`, while the raw target line remains `(batch_size, 12, num_features)`.


In [6]:
#checker
dlinear_batch, dlinear_prediction = preview_shapes(dlinear_cfg, "DLinear")


DLinear raw inputs shape:        (32, 96, 7)
DLinear raw target shape:        (32, 12, 7)
DLinear processed inputs shape:  (32, 96, 7)
DLinear target shape:            (32, 6, 7)
DLinear prediction shape:        (32, 6, 7)


## 7. Train the DLinear

This is the first training run. Leave `RUN_DLinear_TRAINING` as `False` while editing or shape-checking, then switch it to `True` when you are ready to train.


In [7]:
#training
RUN_DLinear_TRAINING = False
RUN_DLinear_12_STEP_TRAINING = False

if RUN_DLinear_TRAINING:
    dlinear_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "DLinear",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_1_6",
    )
    print("Training split DLinear in:", dlinear_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(dlinear_cfg)
else:
    print("DLinear split-horizon training skipped. Set RUN_DLinear_TRAINING = True to train.")

if RUN_DLinear_12_STEP_TRAINING:
    dlinear_full_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "DLinear",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    )
    print("Training 12-step DLinear in:", dlinear_full_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(dlinear_full_cfg)
else:
    print("DLinear 12-step training skipped. Set RUN_DLinear_12_STEP_TRAINING = True to train.")


DLinear split-horizon training skipped. Set RUN_DLinear_TRAINING = True to train.
DLinear 12-step training skipped. Set RUN_DLinear_12_STEP_TRAINING = True to train.


## 8. Transformer Model

After the DLinear shape check works, use the repo's existing `iTransformerForForecasting`. It receives the same `[batch_size, input_len, num_features]` input and returns `[batch_size, 6, num_features]`. In this split-horizon hybrid, the Transformer is responsible for forecast steps 7-12.


In [8]:
#transfoemr model build it
transformer_model_config = iTransformerConfig(
    input_len=INPUT_LEN,
    output_len=SPLIT_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=64,
    intermediate_size=128,
    n_heads=4,
    num_layers=2,
    dropout=0.1,
    use_revin=True,
)

transformer_cfg = BasicTSForecastingConfig(
    model=iTransformerForForecasting,
    model_config=transformer_model_config,
    taskflow=SplitHorizonForecastingTaskFlow(slice(SPLIT_OUTPUT_LEN, FULL_OUTPUT_LEN)),
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/iTransformerForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_7_12",
    **SHARED_CONFIG,
)

# Standalone Transformer trained to forecast all 12 steps.
transformer_full_model_config = iTransformerConfig(
    input_len=INPUT_LEN,
    output_len=FULL_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=64,
    intermediate_size=128,
    n_heads=4,
    num_layers=2,
    dropout=0.1,
    use_revin=True,
)

transformer_full_cfg = BasicTSForecastingConfig(
    model=iTransformerForForecasting,
    model_config=transformer_full_model_config,
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/iTransformerForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    **FULL_12_STEP_CONFIG,
)

transformer_cfg, transformer_full_cfg


(BasicTSForecastingConfig(model=<class 'basicts.models.iTransformer.arch.itransformer_arch.iTransformerForForecasting'>, model_config=iTransformerConfig(input_len=96, output_len=6, num_features=7, num_classes=None, hidden_size=64, n_heads=4, intermediate_size=128, hidden_act='gelu', num_layers=2, dropout=0.1, use_revin=True, output_attentions=False), dataset_name='ETTh1', taskflow=<__main__.SplitHorizonForecastingTaskFlow object at 0x0000023F4DBC4CD0>, callbacks=[], gpus=None, gpu_num=0, seed=42, dataset_type=<class 'basicts.data.tsf_dataset.BasicTSForecastingDataset'>, dataset_params={'input_len': 96, 'output_len': 12, 'use_timestamps': False, 'memmap': False, 'dataset_name': 'ETTh1'}, batch_size=32, null_val=nan, null_to_num=0.0, scaler=<class 'basicts.scaler.z_score_scaler.ZScoreScaler'>, norm_each_channel=True, rescale=False, ddp_find_unused_parameters=False, compile_model=False, metrics=['MAE', 'MSE'], target_metric='MAE', best_metric='min', num_epochs=5, num_steps=None, loss='MAE

## 9. Transformer Shape Test

Run this after the DLinear section works. It uses the same shared data pipeline and checks that the Transformer predicts only its 6-step split-horizon target.


In [9]:
#check the shapes
transformer_batch, transformer_prediction = preview_shapes(transformer_cfg, "Transformer")


Transformer raw inputs shape:        (32, 96, 7)
Transformer raw target shape:        (32, 12, 7)
Transformer processed inputs shape:  (32, 96, 7)
Transformer target shape:            (32, 6, 7)
Transformer prediction shape:        (32, 6, 7)


## 10. Train the Transformer

Use the same dataset, scaler, preprocessing, and `input_len` as the DLinear. The shared dataset still contains the full 12-step target window, but the Transformer taskflow slices that target to steps 7-12.


In [10]:
#train trasnfoemr
RUN_TRANSFORMER_TRAINING = False
RUN_TRANSFORMER_12_STEP_TRAINING = False

if RUN_TRANSFORMER_TRAINING:
    transformer_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "iTransformerForForecasting",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_7_12",
    )
    print("Training split Transformer in:", transformer_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(transformer_cfg)
else:
    print("Transformer split-horizon training skipped. Set RUN_TRANSFORMER_TRAINING = True after the DLinear works.")

if RUN_TRANSFORMER_12_STEP_TRAINING:
    transformer_full_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "iTransformerForForecasting",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    )
    print("Training 12-step Transformer in:", transformer_full_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(transformer_full_cfg)
else:
    print("Transformer 12-step training skipped. Set RUN_TRANSFORMER_12_STEP_TRAINING = True to train.")


Transformer split-horizon training skipped. Set RUN_TRANSFORMER_TRAINING = True after the DLinear works.
Transformer 12-step training skipped. Set RUN_TRANSFORMER_12_STEP_TRAINING = True to train.


## 11. Hybrid Prediction: DLinear Steps 1-6, Transformer Steps 7-12

This is a true split-horizon hybrid. The DLinear predicts only the first 6 forecast steps, the Transformer predicts only the next 6 forecast steps from the same input window, and the final hybrid prediction is the time-axis concatenation of those two 6-step outputs.


In [11]:
import numpy as np

#checks that the dlinear and transformer both used the same batch
assert torch.equal(dlinear_batch["inputs"], transformer_batch["inputs"])

# Split-horizon hybrid: DLinear predicts steps 1-6, Transformer predicts steps 7-12.
dlinear_pred = dlinear_prediction
transformer_pred = transformer_prediction
#this combines the two step prediction into one 12 step hybrui prediction
hybrid_pred = torch.cat([dlinear_pred, transformer_pred], dim=1)
#combines the targests together
hybrid_targets = torch.cat([dlinear_batch["targets"], transformer_batch["targets"]], dim=1)
# checks 
print("DLinear prediction shape:", tuple(dlinear_pred.shape))
print("Transformer prediction shape:", tuple(transformer_pred.shape))
print("Hybrid prediction shape:", tuple(hybrid_pred.shape))
print("Target shape:", tuple(hybrid_targets.shape))

assert tuple(dlinear_pred.shape) == (BATCH_SIZE, SPLIT_OUTPUT_LEN, NUM_FEATURES)
assert tuple(transformer_pred.shape) == (BATCH_SIZE, SPLIT_OUTPUT_LEN, NUM_FEATURES)
assert tuple(hybrid_pred.shape) == (BATCH_SIZE, FULL_OUTPUT_LEN, NUM_FEATURES)
assert tuple(hybrid_targets.shape) == (BATCH_SIZE, FULL_OUTPUT_LEN, NUM_FEATURES)

#find the last saved prediction
def latest_prediction_file(cfg):
    prediction_files = sorted(
        Path(cfg.ckpt_save_dir).rglob("test_results/prediction.npy"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not prediction_files:
        raise FileNotFoundError(
            f"No prediction.npy found under {cfg.ckpt_save_dir}. Train/evaluate this model with save_results=True first."
        )
    return prediction_files[0]

#load the prediction arrays
def load_basicts_array(path, shape):
    # BasicTS writes these files as raw memmaps, even though the file names end in .npy.
    array = np.memmap(path, dtype=np.float32, mode="r", shape=shape)
    return np.asarray(array)

#loads the largets
def load_basicts_prediction_and_targets(cfg, output_len):
    prediction_path = latest_prediction_file(cfg)
    targets_path = prediction_path.parent / "targets.npy"
    if not targets_path.exists():
        raise FileNotFoundError(f"No targets.npy found next to {prediction_path}")

    test_dataset = Builder._build_dataset(cfg, BasicTSMode.TEST)
    shape = (len(test_dataset), output_len, NUM_FEATURES)
    prediction = load_basicts_array(prediction_path, shape)
    targets = load_basicts_array(targets_path, shape)
    return prediction, targets


dlinear_test_pred, dlinear_test_targets = load_basicts_prediction_and_targets(dlinear_cfg, SPLIT_OUTPUT_LEN)
transformer_test_pred, transformer_test_targets = load_basicts_prediction_and_targets(transformer_cfg, SPLIT_OUTPUT_LEN)
#bombines the test predictions
hybrid_test_pred = np.concatenate([dlinear_test_pred, transformer_test_pred], axis=1)
hybrid_test_targets = np.concatenate([dlinear_test_targets, transformer_test_targets], axis=1)

assert dlinear_test_pred.shape == dlinear_test_targets.shape
assert transformer_test_pred.shape == transformer_test_targets.shape
assert dlinear_test_pred.shape == transformer_test_pred.shape
assert hybrid_test_pred.shape == hybrid_test_targets.shape
assert hybrid_test_pred.shape[1] == FULL_OUTPUT_LEN

mixed_output_dir = Path("checkpoints") / RUN_TAG
mixed_output_dir.mkdir(parents=True, exist_ok=True)
hybrid_save_path = mixed_output_dir / "hybrid_split_horizon_dlinear_steps_1_6_transformer_steps_7_12_ETTh1_96_12_prediction.npy"
np.save(hybrid_save_path, hybrid_test_pred)
print(f"Saved fixed split hybrid prediction: {hybrid_save_path}")

DLinear prediction shape: (32, 6, 7)
Transformer prediction shape: (32, 6, 7)
Hybrid prediction shape: (32, 12, 7)
Target shape: (32, 12, 7)
Saved fixed split hybrid prediction: checkpoints\mixed_4\hybrid_split_horizon_dlinear_steps_1_6_transformer_steps_7_12_ETTh1_96_12_prediction.npy


## 12. MAE/MSE Comparison

This cell computes MAE/MSE directly from the saved predictions and targets. The DLinear is compared only against target steps 1-6, the Transformer only against target steps 7-12, and the hybrid against the full 12-step target.


In [12]:
#use math to compute metrics
def compute_metrics(prediction, targets):
    return {
        "MAE": float(np.mean(np.abs(prediction - targets))),
        "MSE": float(np.mean((prediction - targets) ** 2)),
    }

def latest_metrics_file(cfg):
    metrics_files = sorted(
        Path(cfg.ckpt_save_dir).rglob("test_metrics.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not metrics_files:
        raise FileNotFoundError(f"No test_metrics.json found under {cfg.ckpt_save_dir}. Train this model first.")
    return metrics_files[0]


def load_test_metrics(cfg):
    metrics_path = latest_metrics_file(cfg)
    with metrics_path.open("r", encoding="utf-8") as f:
        metrics = json.load(f)
    return metrics.get("overall", metrics)


# Compare the split models, hybrid, and standalone 12-step models.
comparison = {
    "DLinear split steps 1-6": compute_metrics(dlinear_test_pred, dlinear_test_targets),
    "Transformer split steps 7-12": compute_metrics(transformer_test_pred, transformer_test_targets),
    "Hybrid split steps 1-12": compute_metrics(hybrid_test_pred, hybrid_test_targets),
    "DLinear full steps 1-12": load_test_metrics(dlinear_full_cfg),
    "Transformer full steps 1-12": load_test_metrics(transformer_full_cfg),
}

print("DLinear target shape:", dlinear_test_targets.shape)
print("Transformer target shape:", transformer_test_targets.shape)
print("Hybrid target shape:", hybrid_test_targets.shape)
print("DLinear full 12-step metrics file:", latest_metrics_file(dlinear_full_cfg))
print("Transformer full 12-step metrics file:", latest_metrics_file(transformer_full_cfg))

for model_name, metrics in comparison.items():
    print(model_name)
    for metric_name in ["MAE", "MSE"]:
        print(f"  {metric_name}: {metrics[metric_name]:.6f}")

DLinear target shape: (2773, 6, 7)
Transformer target shape: (2773, 6, 7)
Hybrid target shape: (2773, 12, 7)
DLinear full 12-step metrics file: checkpoints\mixed_4\DLinear\ETTh1_96_steps_1_12\20260702_012150\51c7e8a94d5965d70a936858a20906c5\test_metrics.json
Transformer full 12-step metrics file: checkpoints\mixed_4\iTransformerForForecasting\ETTh1_96_steps_1_12\20260702_012100\25ba3ab815b51a6c9228d6a8b35bab77\test_metrics.json
DLinear split steps 1-6
  MAE: 0.300875
  MSE: 0.232997
Transformer split steps 7-12
  MAE: 0.358287
  MSE: 0.323265
Hybrid split steps 1-12
  MAE: 0.329581
  MSE: 0.278131
DLinear full steps 1-12
  MAE: 0.331306
  MSE: 0.282716
Transformer full steps 1-12
  MAE: 0.332324
  MSE: 0.277150


## 13. Efficiency Comparison

This cell compares model size and average batch prediction time. The DLinear timing is for its 6-step prediction, the Transformer timing is for its 6-step prediction, and the hybrid timing is the sum of running both 6-step models once.


In [13]:

import time

#counts model size
def count_trainable_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)

#takes one batch and runs the model many times
def time_model_prediction(model, batch, expected_output_len, repeats=50, warmup=5):
    #measure how fast a model makes predictions in one batch
    # check if its on cpu or gpu
    device = next(model.parameters()).device
    # moves input to the same device
    inputs = batch["inputs"].to(device)
    model.eval()

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(inputs)
    #predicts the batch 50 times
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(repeats):
            prediction = model(inputs)
    end = time.perf_counter()

    if isinstance(prediction, dict):
        prediction = prediction["prediction"]
    assert tuple(prediction.shape) == (inputs.size(0), expected_output_len, NUM_FEATURES)
    return (end - start) / repeats


dlinear_efficiency_model = dlinear_cfg.model(dlinear_cfg.model_config)
transformer_efficiency_model = transformer_cfg.model(transformer_cfg.model_config)
dlinear_full_efficiency_model = dlinear_full_cfg.model(dlinear_full_cfg.model_config)
transformer_full_efficiency_model = transformer_full_cfg.model(transformer_full_cfg.model_config)

dlinear_params = count_trainable_parameters(dlinear_efficiency_model)
transformer_params = count_trainable_parameters(transformer_efficiency_model)
dlinear_full_params = count_trainable_parameters(dlinear_full_efficiency_model)
transformer_full_params = count_trainable_parameters(transformer_full_efficiency_model)
hybrid_params = dlinear_params + transformer_params

dlinear_time = time_model_prediction(dlinear_efficiency_model, dlinear_batch, SPLIT_OUTPUT_LEN)
transformer_time = time_model_prediction(transformer_efficiency_model, transformer_batch, SPLIT_OUTPUT_LEN)
dlinear_full_time = time_model_prediction(dlinear_full_efficiency_model, dlinear_batch, FULL_OUTPUT_LEN)
transformer_full_time = time_model_prediction(transformer_full_efficiency_model, transformer_batch, FULL_OUTPUT_LEN)
hybrid_time = dlinear_time + transformer_time

print("DLinear parameters:", dlinear_params)
print("Transformer parameters:", transformer_params)
print("Hybrid parameters:", hybrid_params)
print("DLinear full 12-step parameters:", dlinear_full_params)
print("Transformer full 12-step parameters:", transformer_full_params)

print(f"DLinear 6-step avg prediction time: {dlinear_time:.6f} seconds")
print(f"Transformer 6-step avg prediction time: {transformer_time:.6f} seconds")
print(f"Hybrid avg prediction time: {hybrid_time:.6f} seconds")
print(f"DLinear full 12-step avg prediction time: {dlinear_full_time:.6f} seconds")
print(f"Transformer full 12-step avg prediction time: {transformer_full_time:.6f} seconds")

rows = [
    ("DLinear split 1-6", comparison["DLinear split steps 1-6"], dlinear_params, dlinear_time),
    ("Transformer split 7-12", comparison["Transformer split steps 7-12"], transformer_params, transformer_time),
    ("Hybrid split 1-12", comparison["Hybrid split steps 1-12"], hybrid_params, hybrid_time),
    ("DLinear full 1-12", comparison["DLinear full steps 1-12"], dlinear_full_params, dlinear_full_time),
    ("Transformer full 1-12", comparison["Transformer full steps 1-12"], transformer_full_params, transformer_full_time),
]

print(f"{'Model':<14} {'MAE':>10} {'MSE':>10} {'Params':>12} {'Avg batch sec':>15}")
print("-" * 65)
for model_name, metrics, params, avg_time in rows:
    print(f"{model_name:<14} {metrics['MAE']:>10.6f} {metrics['MSE']:>10.6f} {params:>12,} {avg_time:>15.6f}")


DLinear parameters: 1164
Transformer parameters: 73670
Hybrid parameters: 74834
DLinear full 12-step parameters: 2328
Transformer full 12-step parameters: 74060
DLinear 6-step avg prediction time: 0.000755 seconds
Transformer 6-step avg prediction time: 0.004355 seconds
Hybrid avg prediction time: 0.005110 seconds
DLinear full 12-step avg prediction time: 0.000519 seconds
Transformer full 12-step avg prediction time: 0.003948 seconds
Model                 MAE        MSE       Params   Avg batch sec
-----------------------------------------------------------------
DLinear split 1-6   0.300875   0.232997        1,164        0.000755
Transformer split 7-12   0.358287   0.323265       73,670        0.004355
Hybrid split 1-12   0.329581   0.278131       74,834        0.005110
DLinear full 1-12   0.331306   0.282716        2,328        0.000519
Transformer full 1-12   0.332324   0.277150       74,060        0.003948


## 14. Selector-Based Hybrid: Hard and Soft Weighted Selection Without Test Leakage

In this section, we move beyond the fixed split-horizon hybrid (DLinear steps 1-6, Transformer steps 7-12) to intelligent selection mechanisms:

1. **Hard Selector**: For each forecast step and feature, choose the model with the lowest validation MAE.
2. **Soft Weighted Selector**: For each forecast step and feature, blend predictions using weights derived from inverse validation MAE.

The important rule: validation data is used to learn the selector, and test data is used only for final evaluation.

In [14]:
# Generate full 12-step validation and test predictions for selector comparison.
# The selector is learned on validation predictions only, then applied to test predictions.

from types import SimpleNamespace

print("Loading full 12-step model predictions for VAL and TEST...")


def latest_best_checkpoint(cfg):
    metric_name = cfg.target_metric.replace("/", "_")
    checkpoint_name = f"{cfg.model.__name__}_best_val_{metric_name}.pt"
    checkpoint_files = sorted(
        Path(cfg.ckpt_save_dir).rglob(checkpoint_name),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not checkpoint_files:
        raise FileNotFoundError(
            f"No {checkpoint_name} found under {cfg.ckpt_save_dir}. Train the model first."
        )
    return checkpoint_files[0]


def predict_full_model_on_mode(cfg, mode):
    train_dataset = Builder._build_dataset(cfg, BasicTSMode.TRAIN)
    eval_dataset = Builder._build_dataset(cfg, mode)
    eval_loader = DataLoader(eval_dataset, batch_size=cfg.batch_size, shuffle=False)

    scaler = Builder._build_scaler(cfg) if cfg.scaler is not None else None
    if scaler is not None:
        scaler.fit(train_dataset.data)

    model = cfg.model(cfg.model_config)
    checkpoint_path = latest_best_checkpoint(cfg)
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    runner = SimpleNamespace(cfg=cfg, scaler=scaler)
    predictions = []
    targets = []

    with torch.no_grad():
        for raw_batch in eval_loader:
            batch = _float_batch(raw_batch)
            batch = cfg.taskflow.preprocess(runner, batch)
            prediction = model(batch["inputs"])
            if isinstance(prediction, dict):
                prediction = prediction["prediction"]
            predictions.append(prediction.cpu().numpy())
            targets.append(batch["targets"].cpu().numpy())

    return np.concatenate(predictions, axis=0), np.concatenate(targets, axis=0), checkpoint_path


dlinear_full_val_pred, dlinear_full_val_targets, dlinear_full_ckpt = predict_full_model_on_mode(
    dlinear_full_cfg,
    BasicTSMode.VAL,
)
transformer_full_val_pred, transformer_full_val_targets, transformer_full_ckpt = predict_full_model_on_mode(
    transformer_full_cfg,
    BasicTSMode.VAL,
)

dlinear_full_test_pred, dlinear_full_test_targets, _ = predict_full_model_on_mode(
    dlinear_full_cfg,
    BasicTSMode.TEST,
)
transformer_full_test_pred, transformer_full_test_targets, _ = predict_full_model_on_mode(
    transformer_full_cfg,
    BasicTSMode.TEST,
)

print(f"DLinear best checkpoint: {dlinear_full_ckpt}")
print(f"Transformer best checkpoint: {transformer_full_ckpt}")
print(f"DLinear VAL pred shape: {dlinear_full_val_pred.shape}")
print(f"Transformer VAL pred shape: {transformer_full_val_pred.shape}")
print(f"DLinear TEST pred shape: {dlinear_full_test_pred.shape}")
print(f"Transformer TEST pred shape: {transformer_full_test_pred.shape}")

assert dlinear_full_val_pred.shape == dlinear_full_val_targets.shape
assert transformer_full_val_pred.shape == transformer_full_val_targets.shape
assert dlinear_full_test_pred.shape == dlinear_full_test_targets.shape
assert transformer_full_test_pred.shape == transformer_full_test_targets.shape
assert dlinear_full_val_pred.shape[1:] == (FULL_OUTPUT_LEN, NUM_FEATURES)
assert dlinear_full_test_pred.shape[1:] == (FULL_OUTPUT_LEN, NUM_FEATURES)

Loading full 12-step model predictions for VAL and TEST...
DLinear best checkpoint: checkpoints\mixed_4\DLinear\ETTh1_96_steps_1_12\20260702_012150\51c7e8a94d5965d70a936858a20906c5\DLinear_best_val_MAE.pt
Transformer best checkpoint: checkpoints\mixed_4\iTransformerForForecasting\ETTh1_96_steps_1_12\20260702_012100\25ba3ab815b51a6c9228d6a8b35bab77\iTransformerForForecasting_best_val_MAE.pt
DLinear VAL pred shape: (2773, 12, 7)
Transformer VAL pred shape: (2773, 12, 7)
DLinear TEST pred shape: (2773, 12, 7)
Transformer TEST pred shape: (2773, 12, 7)


In [ ]:
# Hard Selector: Learn per-step/per-feature choices on VAL, then apply to TEST.

print("\n=== HARD SELECTOR (Validation-Tuned Step-wise Best Model) ===")

dlinear_val_per_step_mae = np.mean(
    np.abs(dlinear_full_val_pred - dlinear_full_val_targets),
    axis=0,
)

transformer_val_per_step_mae = np.mean(
    np.abs(transformer_full_val_pred - transformer_full_val_targets),
    axis=0,
)

print(f"DLinear validation per-step MAE shape: {dlinear_val_per_step_mae.shape}")
print(f"Transformer validation per-step MAE shape: {transformer_val_per_step_mae.shape}")

# hard_selector_mask[step, feature] = 0 for DLinear, 1 for Transformer.
# This is learned only from validation errors.
#make this more generalizable
hard_selector_mask = (
    transformer_val_per_step_mae < dlinear_val_per_step_mae
).astype(int)

hard_selected_pred = np.where(
    hard_selector_mask[np.newaxis, :, :],
    transformer_full_test_pred,
    dlinear_full_test_pred,
)

hard_selected_targets = dlinear_full_test_targets

print(f"Hard selector mask shape: {hard_selector_mask.shape}")
print(f"Hard-selected TEST prediction shape: {hard_selected_pred.shape}")
print(f"Hard-selected TEST targets shape: {hard_selected_targets.shape}")

assert np.allclose(dlinear_full_test_targets, transformer_full_test_targets)
assert hard_selected_pred.shape == hard_selected_targets.shape

print("\nHard selector learned from VAL: Model choices per step (% Transformer):")
for step in range(FULL_OUTPUT_LEN):
    pct_transformer = np.mean(hard_selector_mask[step, :]) * 100
    print(f"  Step {step+1}: {pct_transformer:.1f}% Transformer, {100-pct_transformer:.1f}% DLinear")


=== HARD SELECTOR (Validation-Tuned Step-wise Best Model) ===
DLinear validation per-step MAE shape: (12, 7)
Transformer validation per-step MAE shape: (12, 7)
Hard selector mask shape: (12, 7)
Hard-selected TEST prediction shape: (2773, 12, 7)
Hard-selected TEST targets shape: (2773, 12, 7)

Hard selector learned from VAL: Model choices per step (% Transformer):
  Step 1: 0.0% Transformer, 100.0% DLinear
  Step 2: 71.4% Transformer, 28.6% DLinear
  Step 3: 71.4% Transformer, 28.6% DLinear
  Step 4: 100.0% Transformer, 0.0% DLinear
  Step 5: 100.0% Transformer, 0.0% DLinear
  Step 6: 100.0% Transformer, 0.0% DLinear
  Step 7: 100.0% Transformer, 0.0% DLinear
  Step 8: 100.0% Transformer, 0.0% DLinear
  Step 9: 85.7% Transformer, 14.3% DLinear
  Step 10: 85.7% Transformer, 14.3% DLinear
  Step 11: 85.7% Transformer, 14.3% DLinear
  Step 12: 85.7% Transformer, 14.3% DLinear


In [16]:
# Soft Weighted Selector: Learn inverse-MAE weights on VAL, then apply to TEST.

print("\n=== SOFT WEIGHTED SELECTOR (Validation-Tuned Weighted Average) ===")

epsilon = 1e-6

dlinear_weights = 1.0 / (dlinear_val_per_step_mae + epsilon)
transformer_weights = 1.0 / (transformer_val_per_step_mae + epsilon)

total_weights = dlinear_weights + transformer_weights
dlinear_weights_normalized = dlinear_weights / total_weights
transformer_weights_normalized = transformer_weights / total_weights

weighted_pred = (
    dlinear_weights_normalized[np.newaxis, :, :] * dlinear_full_test_pred
    + transformer_weights_normalized[np.newaxis, :, :] * transformer_full_test_pred
)

weighted_targets = dlinear_full_test_targets

print(f"DLinear validation-derived weights shape: {dlinear_weights_normalized.shape}")
print(f"Transformer validation-derived weights shape: {transformer_weights_normalized.shape}")
print(f"Weighted TEST prediction shape: {weighted_pred.shape}")
print(f"Weighted TEST targets shape: {weighted_targets.shape}")

assert weighted_pred.shape == weighted_targets.shape

print("\nSoft weights learned from VAL, averaged across features:")
print(f"{'Step':<6} {'DLinear Weight':<12} {'Transformer Weight':<18}")
print("-" * 36)

for step in range(FULL_OUTPUT_LEN):
    dlinear_step_weight = np.mean(dlinear_weights_normalized[step, :])
    transformer_step_weight = np.mean(transformer_weights_normalized[step, :])
    print(f"{step+1:<6} {dlinear_step_weight:<12.4f} {transformer_step_weight:<18.4f}")


=== SOFT WEIGHTED SELECTOR (Validation-Tuned Weighted Average) ===
DLinear validation-derived weights shape: (12, 7)
Transformer validation-derived weights shape: (12, 7)
Weighted TEST prediction shape: (2773, 12, 7)
Weighted TEST targets shape: (2773, 12, 7)

Soft weights learned from VAL, averaged across features:
Step   DLinear Weight Transformer Weight
------------------------------------
1      0.5154       0.4846            
2      0.4998       0.5002            
3      0.4929       0.5071            
4      0.4885       0.5115            
5      0.4864       0.5136            
6      0.4868       0.5132            
7      0.4885       0.5115            
8      0.4897       0.5103            
9      0.4923       0.5077            
10     0.4929       0.5071            
11     0.4938       0.5062            
12     0.4956       0.5044            


In [17]:
# Comprehensive metrics table.
# The selector hybrids were tuned on VAL and are evaluated here on TEST only.

print("\n=== COMPREHENSIVE TEST METRICS COMPARISON ===\n")

all_comparison = {
    "Transformer split steps 7-12": compute_metrics(transformer_test_pred, transformer_test_targets),
    "Fixed split hybrid 1-12": compute_metrics(hybrid_test_pred, hybrid_test_targets),
    "DLinear full steps 1-12": compute_metrics(dlinear_full_test_pred, dlinear_full_test_targets),
    "Transformer full steps 1-12": compute_metrics(transformer_full_test_pred, transformer_full_test_targets),
    "Hard selector hybrid 1-12": compute_metrics(hard_selected_pred, hard_selected_targets),
    "Soft weighted hybrid 1-12": compute_metrics(weighted_pred, weighted_targets),
}

print(f"{'Model':<35} {'MAE':>12} {'MSE':>12}")
print("-" * 60)
for model_name in [
    "Transformer split steps 7-12",
    "Fixed split hybrid 1-12",
    "DLinear full steps 1-12",
    "Transformer full steps 1-12",
    "Hard selector hybrid 1-12",
    "Soft weighted hybrid 1-12",
]:
    metrics = all_comparison[model_name]
    mae = metrics["MAE"]
    mse = metrics["MSE"]
    print(f"{model_name:<35} {mae:>12.6f} {mse:>12.6f}")

best_mae_model = min(all_comparison.items(), key=lambda x: x[1]["MAE"])
best_mse_model = min(all_comparison.items(), key=lambda x: x[1]["MSE"])

print("\n" + "=" * 60)
print(f"Best TEST MAE: {best_mae_model[0]:<30} {best_mae_model[1]['MAE']:.6f}")
print(f"Best TEST MSE: {best_mse_model[0]:<30} {best_mse_model[1]['MSE']:.6f}")


=== COMPREHENSIVE TEST METRICS COMPARISON ===

Model                                        MAE          MSE
------------------------------------------------------------
Transformer split steps 7-12            0.358287     0.323265
Fixed split hybrid 1-12                 0.329581     0.278131
DLinear full steps 1-12                 0.331306     0.282716
Transformer full steps 1-12             0.332324     0.277150
Hard selector hybrid 1-12               0.331580     0.277364
Soft weighted hybrid 1-12               0.326538     0.273048

Best TEST MAE: Soft weighted hybrid 1-12      0.326538
Best TEST MSE: Soft weighted hybrid 1-12      0.273048


In [18]:
# Save selector hybrid predictions to the mixed run folder in checkpoints/.

print("\n=== SAVING LEAKAGE-SAFE SELECTOR PREDICTIONS ===\n")

mixed_output_dir = Path("checkpoints") / RUN_TAG
mixed_output_dir.mkdir(parents=True, exist_ok=True)

hard_selector_save_path = mixed_output_dir / "hard_selector_val_tuned_hybrid_dlinear_transformer_ETTh1_96_12_prediction.npy"
np.save(hard_selector_save_path, hard_selected_pred)
print(f"Saved hard selector predictions: {hard_selector_save_path}")
print(f"  Shape: {hard_selected_pred.shape}")

weighted_selector_save_path = mixed_output_dir / "soft_weighted_val_tuned_hybrid_dlinear_transformer_ETTh1_96_12_prediction.npy"
np.save(weighted_selector_save_path, weighted_pred)
print(f"Saved soft weighted predictions: {weighted_selector_save_path}")
print(f"  Shape: {weighted_pred.shape}")

weights_info = {
    "selector_fit_split": "validation",
    "final_evaluation_split": "test",
    "dlinear_weights": dlinear_weights_normalized.tolist(),
    "transformer_weights": transformer_weights_normalized.tolist(),
    "hard_selector_mask": hard_selector_mask.tolist(),
    "description": "Hard selector mask: 0=DLinear, 1=Transformer. Soft weights are normalized inverse validation MAE per step and feature.",
}
weights_save_path = mixed_output_dir / "selector_metadata_val_tuned_ETTh1_96_12.json"
with weights_save_path.open("w", encoding="utf-8") as f:
    json.dump(weights_info, f, indent=2)
print(f"Saved selector metadata: {weights_save_path}")

hybrid_metrics_save_path = mixed_output_dir / "hybrid_metrics_ETTh1_96_12.json"
with hybrid_metrics_save_path.open("w", encoding="utf-8") as f:
    json.dump(all_comparison, f, indent=2)
print(f"Saved hybrid metrics: {hybrid_metrics_save_path}")

print("All leakage-safe selector predictions and metadata saved successfully.")


=== SAVING LEAKAGE-SAFE SELECTOR PREDICTIONS ===

Saved hard selector predictions: checkpoints\mixed_4\hard_selector_val_tuned_hybrid_dlinear_transformer_ETTh1_96_12_prediction.npy
  Shape: (2773, 12, 7)
Saved soft weighted predictions: checkpoints\mixed_4\soft_weighted_val_tuned_hybrid_dlinear_transformer_ETTh1_96_12_prediction.npy
  Shape: (2773, 12, 7)
Saved selector metadata: checkpoints\mixed_4\selector_metadata_val_tuned_ETTh1_96_12.json
Saved hybrid metrics: checkpoints\mixed_4\hybrid_metrics_ETTh1_96_12.json
All leakage-safe selector predictions and metadata saved successfully.


In [19]:
# Summary and interpretation of selector approaches

print("\n" + "="*70)
print("LEAKAGE-SAFE SELECTOR-BASED HYBRID SUMMARY")
print("="*70)

print("""
Three hybrid approaches have been compared:

1. FIXED SPLIT HYBRID
   - DLinear forecasts steps 1-6
   - Transformer forecasts steps 7-12
   - Simple concatenation; uses pre-trained split-horizon models

2. HARD SELECTOR HYBRID
   - For each forecast step and feature, choose the model with the lowest validation MAE
   - The selector mask is frozen before test evaluation
   - File: hard_selector_val_tuned_hybrid_dlinear_transformer_ETTh1_96_12_prediction.npy

3. SOFT WEIGHTED HYBRID
   - For each forecast step and feature, weight-average both model predictions
   - Weights are based on inverse validation MAE, not test MAE
   - File: soft_weighted_val_tuned_hybrid_dlinear_transformer_ETTh1_96_12_prediction.npy

Leakage control:
- Validation data chooses the selector mask and weights
- Test data is used only once for final metrics
- Do not recompute selector choices from test targets
""")

print("="*70)
print("Selector-based hybrid implementation complete.")
print("="*70)


LEAKAGE-SAFE SELECTOR-BASED HYBRID SUMMARY

Three hybrid approaches have been compared:

1. FIXED SPLIT HYBRID
   - DLinear forecasts steps 1-6
   - Transformer forecasts steps 7-12
   - Simple concatenation; uses pre-trained split-horizon models

2. HARD SELECTOR HYBRID
   - For each forecast step and feature, choose the model with the lowest validation MAE
   - The selector mask is frozen before test evaluation
   - File: hard_selector_val_tuned_hybrid_dlinear_transformer_ETTh1_96_12_prediction.npy

3. SOFT WEIGHTED HYBRID
   - For each forecast step and feature, weight-average both model predictions
   - Weights are based on inverse validation MAE, not test MAE
   - File: soft_weighted_val_tuned_hybrid_dlinear_transformer_ETTh1_96_12_prediction.npy

Leakage control:
- Validation data chooses the selector mask and weights
- Test data is used only once for final metrics
- Do not recompute selector choices from test targets

Selector-based hybrid implementation complete.
